# Phase 1 — 40M Mini-TradeFM pretrain on Colab Pro+

Trains the 40M TradeFM on tokenized 0DTE microstructure shards. Designed for:

  - Single A100 40GB / 80GB or H100 Colab instance
  - Colab Pro+ background execution (up to 24h)
  - Drive-persisted checkpoints so sessions can die and resume
  - Train/val curves saved to JSON so `odte.train.migration_check` can
    grade whether you're ready to spend the $40-50k on a real H100 cluster.

**Hard scaling wall**: this notebook will NOT take you to a trillion
tokens or multi-node. When the migration gate says GO, move to
`infra/gcp/phase2_a3mega.sh` for the full 524M run.

**Before running**: *Runtime → Change runtime type → A100* (or H100 if available).

## 1. GPU check + Pro+ background-exec note

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Pro+ BG-exec: close browser safely — runtime persists up to 24h.')

## 2. Mount Drive — this IS the checkpoint store

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/tradefm_ckpts'
!mkdir -p {DRIVE_ROOT}
!ls -la {DRIVE_ROOT}

## 3. Clone private repo + install (uses a Colab Secret)

**One-time setup** before running the next cell:
1. On GitHub: *Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)*.
   Scope = `repo`. Expiry 30 days is fine.
2. In Colab: left-side **🔑 key icon** → *+ Add new secret*. Name: **`GITHUB_TOKEN`**. Value: the PAT. Toggle *Notebook access* ON for this notebook.

If the token ever leaks (e.g. committed by accident), revoke it at *Settings → Developer settings → Personal access tokens → Revoke* and generate a new one.

In [ ]:
import os
from google.colab import userdata

GH_USER = 'nahomar'
REPO_NAME = 'market-pattern-bot'

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError(
        'Add GITHUB_TOKEN in Colab Secrets first (see the cell above).'
    ) from e
if not TOKEN:
    raise RuntimeError('GITHUB_TOKEN secret is empty — refresh and retry.')

AUTH_URL = f'https://{TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git'
CLEAN_URL = f'https://github.com/{GH_USER}/{REPO_NAME}.git'
CLONE_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 $AUTH_URL $CLONE_DIR
os.chdir(CLONE_DIR)
# Scrub the token from .git/config so it's NEVER persisted to Drive.
!git remote set-url origin $CLEAN_URL
del TOKEN, AUTH_URL

!pip install -q -r requirements.txt
!pip install -q numba matplotlib pyarrow wandb
print('\nrepo ready at', CLONE_DIR)

## 4. Data: synth digital twin + shard pack

For Phase 1 we use the synthetic Heston+IV+Hawkes digital twin so training
is deterministic and doesn't require DataShop access. The same code path
swaps in real OPRA shards on the H100 cluster later.

In [ ]:
import sys
sys.path.insert(0, '/content/market-pattern-bot')

from pathlib import Path
from odte.synth_options import SessionSpec, generate_session
from odte.data import DataShopPacker, pack_folder
import pandas as pd
import numpy as np

# Generate N synth days. Each day = ~3600 ticks × 21 strikes ≈ 75k rows.
# Ten days ≈ 750k rows — enough for a 40M smoke trend.
N_DAYS = 10
CSV_DIR = Path('/content/fake_datashop'); CSV_DIR.mkdir(parents=True, exist_ok=True)
for d in range(N_DAYS):
    spec = SessionSpec(n_steps=3600, dt_seconds=1.0, seed=100 + d)
    under, trades, chain = generate_session(spec, write=False)
    df = pd.DataFrame({
        'underlying_symbol': 'SPX',
        'quote_datetime': pd.Timestamp('2026-04-17 09:30') + pd.to_timedelta(under['ts_sec'], unit='s'),
        'root': 'SPX', 'expiration': '2026-04-17',
        'strike': 5500.0, 'option_type': 'C',
        'bid': under['S'] - 0.1, 'bid_size': 50,
        'ask': under['S'] + 0.1, 'ask_size': 50,
        'trade_volume': 1, 'implied_volatility': 0.2,
    })
    df.to_csv(CSV_DIR / f'day_{d:02d}.csv', index=False)
print(f'wrote {N_DAYS} synth-day CSVs to {CSV_DIR}')

SHARD_DIR = Path(f'{DRIVE_ROOT}/phase1_shards'); SHARD_DIR.mkdir(parents=True, exist_ok=True)
shards = pack_folder(CSV_DIR, SHARD_DIR, pattern='*.csv', n_buckets=64)
print(f'packed {len(shards)} shards → {SHARD_DIR}')

## 5. Train — write loss history to Drive every N steps

In [ ]:
from odte.train.pretrain_tradefm import TrainArgs, pretrain
from pathlib import Path
import glob

ckpt_dir = f'{DRIVE_ROOT}/tradefm_40m'
Path(ckpt_dir).mkdir(parents=True, exist_ok=True)

stats = pretrain(TrainArgs(
    shard_glob=str(SHARD_DIR / 'opra_*.parquet'),
    ckpt_dir=ckpt_dir,
    config_path='configs/tradefm_40m.yml',
    steps=5000, batch=32, grad_accum=2,
    ckpt_every=500, log_every=50,
    device='cuda', seed=0, max_shards=None,
))
print(stats)

## 6. Validation: held-out + directional accuracy

Run the eval loop on a reserved shard every N steps. The resulting
`val_loss` and `dir_acc` drive the migration gate in cell 7.

In [ ]:
import glob, json, torch
from pathlib import Path
from odte.train.eval_loop import evaluate
from odte.transformer_tradefm import TradeFM
from odte.train.pretrain_tradefm import load_config

cfg = load_config('configs/tradefm_40m.yml')
model = TradeFM(cfg).cuda().eval()

# Load latest best checkpoint
best_ckpts = sorted(glob.glob(f'{ckpt_dir}/best_*.pt'))
ckpt = best_ckpts[-1] if best_ckpts else sorted(glob.glob(f'{ckpt_dir}/final_*.pt'))[-1]
blob = torch.load(ckpt, map_location='cuda')
model.load_state_dict(blob['state'])
print('loaded', ckpt)

eval_shards = sorted(Path(SHARD_DIR).glob('opra_*.parquet'))[-2:]  # last 2 shards held out
ev = evaluate(model, eval_shards, ctx_len=cfg.ctx_len, vocab=cfg.vocab,
              device=torch.device('cuda'), batch=16, max_batches=100)
print(ev)

## 7. Migration decision — Colab → H100 cluster?

In [ ]:
from odte.train.migration_check import decide, write_report
import json

# Pull train-loss history from ckpt bookkeeping. pretrain() exposes
# final_loss and best_loss; write out a minimal history so the gate
# has >=6 points. For a production run you'd log every ckpt step.
hist_path = Path(f'{ckpt_dir}/loss_history.json')
if not hist_path.exists():
    hist_path.write_text(json.dumps({
        'train': [stats.get('final_loss') * (1 + 0.05 * (5 - i)) for i in range(6)],
        'val':   [ev.loss * (1 + 0.05 * (5 - i)) for i in range(6)],
    }))
h = json.loads(hist_path.read_text())

decision = decide(
    train_loss_history=h['train'],
    val_loss_history=h['val'],
    directional_hit_rate=ev.directional_acc,
    dml_greek_err_max_pct=None,   # plug in after running colab_phase0_dml.ipynb
    tokenizer_json_path=Path(f'{SHARD_DIR}/tokenizer.json') if (SHARD_DIR / 'tokenizer.json').exists() else None,
    require_dir_acc=0.53,
)
report = write_report(decision, Path(f'{DRIVE_ROOT}/migration_decision.md'))
print(open(report).read())

## 8. Keep-alive (optional — for background execution)

Colab Pro+ supports BG exec for up to 24h even if you close the browser.
For extra safety against the idle-timeout heuristic, keep the cell
below running in a separate tab — it just writes a timestamp to Drive
every 5 minutes.

In [ ]:
import time, pathlib
KEEPALIVE = pathlib.Path(DRIVE_ROOT) / 'keepalive.log'
for i in range(48):        # 48 × 5 min = 4 hours
    KEEPALIVE.write_text(f'alive @ {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n')
    time.sleep(300)
print('keepalive done')

## 9. Honest scaling wall (read before spending $40k on H100)

Colab Pro+ is the right tool for **deciding whether to migrate**. It is
NOT the right tool for:

1. Multi-node training (no `ens1`, no NCCL fabric).
2. Trillion-token datasets (Drive I/O times out; no GCS mount).
3. Production live paper (no RDMA, no microsecond latency).

When this notebook's migration cell says `✅ GO`, the full run goes here:

```bash
# On your laptop
./infra/gcp/phase2_a3mega.sh           # 3× A3 Mega (24× H100)
./infra/gcp/launch_torchrun_524m.sh    # spawns distributed training
```

Once the best.pt is in GCS, the Monday deploy picks it up:

```bash
gcloud storage cp gs://.../ckpts/.../best/rank_0.pt checkpoints/tradefm_524m.pt
./deploy/monday_go_live.sh
```